# 00a · Descarga corregida de microarrays para cuatro subtipos intrínsecos

Endpoint único: **LumA, LumB, HER2-enriched y Basal-like**.

Reglas aplicadas en este notebook:

- solo se conservan tumores con una llamada de subtipo basada en expresión;
- `Basal-like` no se renombra como `TNBC`;
- `HER2-enriched` no se interpreta como HER2 positivo por IHQ;
- se eliminan tejido normal y la clase molecular Normal-like;
- GSE20711 usa `subtypege`, no `subtypeihc`;
- GSE65194 queda fuera del endpoint principal porque sus grupos son IHQ y no subtipos intrínsecos.

El notebook guarda cinco cohortes de microarrays listas para la unión posterior.


In [ ]:
from pathlib import Path
Path('Data').mkdir(parents=True, exist_ok=True)


In [ ]:
import re, requests, pandas as pd, GEOparse as gp
import numpy as np
from collections import defaultdict
import pandas as pd
from typing import Dict, List, Tuple
import os, gzip, shutil, subprocess
from pathlib import Path
import warnings
from pandas.errors import DtypeWarning
warnings.filterwarnings("ignore", category=DtypeWarning)
import io
import contextlib

@contextlib.contextmanager
def _quiet_stdout_stderr():
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

In [ ]:
# ------------------------ helpers ------------------------
# Cache global: RefSeq/Accession -> Symbol
_ACC2SYM_CACHE: dict[str, str] = {}

def _extract_first_accession(x):
    """Devuelve el primer accession tipo NM_/NR_/XM_/XR_/BC_/BX_...; conserva versión (.1)."""
    if pd.isna(x):
        return np.nan
    t = str(x).strip()
    # separadores típicos en anotaciones
    toks = re.split(r"\s*///\s*|[;,]\s*|\s*\|\s*|\s+", t)
    for tok in toks:
        tok = tok.strip()
        if not tok or tok in {"---", "NA", "na"}:
            continue
        # RefSeq/GenBank accession típicos
        if re.match(r"^(NM|NR|XM|XR|BC|BX|AY|AF|DQ|EU|U|X|Y|Z)_[0-9]+(\.[0-9]+)?$", tok):
            return tok
    return np.nan

def _acc_to_symbol_mygene(acc_list, species="human", chunk_size=1000, retries=3, sleep_s=1.0):
    """
    Devuelve dict {accession: SYMBOL}. Usa MyGene.info querymany con scopes refseq/accession.
    """
    try:
        import mygene
    except ImportError as e:
        raise ImportError("Instala mygene: pip install mygene") from e

    mg = mygene.MyGeneInfo()

    acc_list = [str(a) for a in acc_list if pd.notna(a)]
    # cache
    to_query = [a for a in acc_list if a not in _ACC2SYM_CACHE]

    for i in range(0, len(to_query), chunk_size):
        chunk = to_query[i:i+chunk_size]

        last_err = None
        for att in range(retries):
            try:
                res = mg.querymany(
                    chunk,
                    scopes="refseq,accession",
                    fields="symbol",
                    species=species,
                    as_dataframe=False,
                    returnall=False,
                    verbose=False
                )
                for hit in res:
                    if hit.get("notfound"):
                        continue
                    q = str(hit.get("query", "")).strip()
                    sym = hit.get("symbol", None)
                    if q and sym:
                        _ACC2SYM_CACHE[q] = sym
                last_err = None
                break
            except Exception as e:
                last_err = e
                time.sleep(sleep_s * (att + 1))

        # si falla, no abortamos: seguimos con lo que haya en cache
        if last_err is not None:
            pass

    # construir mapping final
    out = {}
    for a in acc_list:
        sym = _ACC2SYM_CACHE.get(a)
        if sym:
            out[a] = sym
    return out

def _find_accession_col(gpl) -> str | None:
    """Detecta columna tipo GB_ACC / RefSeq / accession en gpl.table."""
    if not hasattr(gpl, "table") or gpl.table is None or gpl.table.empty:
        return None
    cols = list(gpl.table.columns)
    # GPL2700 usa exactamente GB_ACC
    if "GB_ACC" in cols:
        return "GB_ACC"
    # Fallback genérico
    for c in cols:
        nc = str(c).strip().lower()
        if nc in {"gb_acc", "refseq", "refseq_id", "accession", "genbank_accession"}:
            return c
    return None
    
def _df_quantiles_for_logcheck(df: pd.DataFrame):
    """
    Devuelve (q0, q25, q50, q75, q99, q100) ignorando NaNs,
    a partir de todos los valores de la matriz.
    """
    if df.empty:
        return None
    arr = df.to_numpy(dtype=float)
    qs = np.nanquantile(arr, [0.0, 0.25, 0.5, 0.75, 0.99, 1.0])
    return qs  # q0, q25, q50, q75, q99, q100

def _needs_log2(df: pd.DataFrame) -> bool:
    """
    Traducción directa del criterio de R:
    LogC <- (qx[5] > 100) || (qx[6] - qx[1] > 50 && qx[2] > 0)
    (con índices 0-based: q99>100  || (q100-q0>50 && q25>0))
    """
    qs = _df_quantiles_for_logcheck(df)
    if qs is None:
        return False
    q0, q25, q50, q75, q99, q100 = qs
    return (q99 > 100) or ((q100 - q0) > 50 and (q25 > 0))

def _apply_log2_like_R(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    df2 = df.copy()
    df2[df2 <= 0] = 1   # equivalente a: ex[ex <= 0] <- 1
    return np.log2(df2.astype(float))


def _first_item(meta: dict, key: str, default=""):
    v = (meta or {}).get(key, default)
    if isinstance(v, (list, tuple)):
        return v[0] if v else default
    return v

def _norm_key(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[\s/\-]+", "_", s)
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s

def _find_gene_col(gpl) -> str | None:
    if not hasattr(gpl, "table") or gpl.table is None or gpl.table.empty:
        return None

    cols = list(gpl.table.columns)

    def norm(c):
        c2 = str(c).strip().lower().replace("_", " ").replace("-", " ")
        return re.sub(r"\s+", " ", c2)

    # 1) Intentar encontrar símbolo (si existiese)
    for c in cols:
        nc = norm(c)
        if re.search(r"\bgene\s*symbol\b", nc):
            return c
    for c in cols:
        nc = norm(c)
        if nc in {"symbol", "gene symbol", "hgnc symbol"} or "gene name" in nc or "gene title" in nc or "gene assignment" in nc:
            return c

    # 2) Fallback: ORF como Entrez Gene ID (caso GPL8542)
    if "ORF" in cols:
        s = pd.to_numeric(gpl.table["ORF"], errors="coerce")
        if s.notna().mean() > 0.90:   # casi todo numérico => muy probablemente Entrez ID
            return "ORF"

    return None

def entrez_to_symbol(entrez_ids, species="human", chunk_size=1000):
    """
    Devuelve dict: { '1': 'A1BG', '10': 'NAT2', ... }
    Usa MyGene.info (API pública).
    """
    url = "https://mygene.info/v3/gene"
    out = {}

    entrez_ids = [str(x) for x in entrez_ids if pd.notna(x)]
    for i in range(0, len(entrez_ids), chunk_size):
        chunk = entrez_ids[i:i+chunk_size]
        r = requests.post(
            url,
            json={"ids": chunk, "fields": "symbol", "species": species},
            timeout=60
        )
        r.raise_for_status()
        data = r.json()

        for item in data:
            # suele venir _id = entrez, y symbol
            eid = str(item.get("_id") or item.get("query") or "")
            sym = item.get("symbol", None)
            if eid and sym:
                out[eid] = sym

    return out


def _extract_first_symbol(x):
    if pd.isna(x): return np.nan
    t = str(x)
    toks = re.split(r"\s*///\s*|[;,]\s*|\s*\|\s*|\s+", t)
    for tok in toks:
        tok = tok.strip()
        if not tok: 
            continue
        if tok in {"---", "NA", "na"}: 
            continue
        if tok.startswith(("NM_","ENSG")):
            continue
        if re.match(r"^[A-Za-z][A-Za-z0-9\-\._]*$", tok):
            return tok
    return np.nan

# ------------------------ metadata (GSE y GSM) ------------------------
def get_gse_metadata(gse) -> pd.DataFrame:
    m = gse.metadata or {}
    row = {
        "gse_accession": gse.name,
        "title":        _first_item(m, "title", ""),
        "summary":      _first_item(m, "summary", ""),
        "overall_design": _first_item(m, "overall_design", ""),
        "type":         _first_item(m, "type", ""),
        "pubmed_id":    _first_item(m, "pubmed_id", ""),
        "submission_date": _first_item(m, "submission_date", ""),
        "status":       _first_item(m, "status", ""),
        "contributor":  ", ".join(m.get("contributor", [])) if "contributor" in m else "",
        "platforms":    ", ".join(sorted(gse.gpls.keys())),
        "n_samples":    len(gse.gsms),
    }
    return pd.DataFrame([row]).set_index("gse_accession")

def expand_characteristics_columns(pheno: pd.DataFrame) -> pd.DataFrame:
    ph = pheno.copy()
    char_cols = [c for c in ph.columns 
                 if re.match(r"characteristics(_ch[12])?(\.\d+)?$", str(c), flags=re.I) 
                 or str(c).lower().startswith("characteristics_ch")]
    extracted = defaultdict(list)
    all_keys = set()
    for idx, row in ph.iterrows():
        kv = {}
        for c in char_cols:
            val = row.get(c, None)
            if pd.isna(val) or str(val).strip() == "":
                continue
            lines = val if isinstance(val, (list, tuple)) else [val]
            flat = []
            for ln in lines:
                if isinstance(ln, str):
                    flat.extend([s for s in re.split(r"\s*;\s*|\s*\|\s*|\n+", ln) if s.strip() != ""])
            for item in flat:
                if ":" in item:
                    k, v = item.split(":", 1)
                    k = _norm_key(k); v = v.strip()
                    if k in kv and kv[k] != v:
                        kv[k] = f"{kv[k]} | {v}"
                    else:
                        kv[k] = v
                else:
                    free_k = "characteristics_free"
                    kv[free_k] = (kv.get(free_k, "") + " | " + str(item).strip()).strip(" |")
        for k, v in kv.items():
            extracted[k].append((idx, v)); all_keys.add(k)
    if not all_keys:
        return ph
    add = pd.DataFrame(index=ph.index, columns=sorted(all_keys), dtype=object)
    for k, pairs in extracted.items():
        for idx, v in pairs:
            add.at[idx, k] = v
    for c in list(add.columns):
        if c in ph.columns:
            add.rename(columns={c: f"{c}_from_char"}, inplace=True)
    return pd.concat([ph, add], axis=1)

def make_sample_metadata(gse) -> pd.DataFrame:
    # base con campos "limpios" directamente de GSM.metadata
    rows = []
    for gsm_id, gsm in gse.gsms.items():
        meta = gsm.metadata or {}
        rows.append({
            "GSM": gsm_id,
            "title": _first_item(meta, "title", ""),
            "organism_ch1": _first_item(meta, "organism_ch1", _first_item(meta, "organism", "")),
            "source_name_ch1": _first_item(meta, "source_name_ch1", _first_item(meta, "source_name", "")),
            "platform_id": _first_item(meta, "platform_id", ""),
            "geo_accession": _first_item(meta, "geo_accession", gsm_id),
            "description": " ".join(meta.get("description", [])) if "description" in meta else "",
            "supplementary_file": "|".join(meta.get("supplementary_file", [])) if "supplementary_file" in meta else "",
        })
    base = pd.DataFrame(rows).set_index("GSM")

    pheno = gse.phenotype_data.copy()
    pheno.index.name = "GSM"

    # 1) join con sufijos para evitar choque de nombres
    merged = base.join(pheno, how="outer", lsuffix="_gsm", rsuffix="_pheno")

    # 2) coalesce de pares duplicados (prioriza *_gsm; si NaN usa *_pheno)
    dupe_cols = ["title","organism_ch1","source_name_ch1","platform_id","geo_accession","description","supplementary_file"]
    for col in dupe_cols:
        cg, cp = f"{col}_gsm", f"{col}_pheno"
        if cg in merged.columns and cp in merged.columns:
            merged[col] = merged[cg].combine_first(merged[cp])
            merged.drop(columns=[cg, cp], inplace=True)
        elif cg in merged.columns:
            merged.rename(columns={cg: col}, inplace=True)
        elif cp in merged.columns:
            merged.rename(columns={cp: col}, inplace=True)

    # 3) expandir characteristics_* a columnas anchas
    merged_expanded = expand_characteristics_columns(merged)

    front = [c for c in ["title","platform_id","organism_ch1","source_name_ch1",
                         "description","supplementary_file"] if c in merged_expanded.columns]
    return merged_expanded[front + [c for c in merged_expanded.columns if c not in front]]

# Cache global opcional para no re-descargar GPLs
# Cache global opcional para no re-descargar GPLs
import time
import numpy as np

# Cache global para no re-consultar Entrez->Symbol
_ENTREZ2SYM_CACHE: dict[str, str] = {}

# Cache global opcional para no re-descargar GPLs
_GPL_CACHE: dict[str, object] = {}

def load_everything_with_metadata(
    gse_id: str,
    destdir: str = "geo",
    platforms: list[str] | None = None,
    annotate_to_gene: bool = True,
    collapse: str = "mean",
    map_entrez_to_symbol: bool = True,     # <-- NUEVO
    drop_unmapped_symbols: bool = True,    # <-- NUEVO: si True, elimina genes sin símbolo
):
    # ---------- 1) Descarga/carga GSE por HTTPS (sin GPL al inicio) ----------
    soft_path = download_gse_family_soft_https(gse_id, destdir=destdir, overwrite=False)
    gse = gp.get_GEO(filepath=soft_path, annotate_gpl=False, silent=True)

    # ---------- helper: descargar GPL .annot.gz por HTTPS ----------
    def _download_gpl_annot_https(gpl_id: str, destdir: str = "geo", overwrite: bool = False) -> str:
        gpl_id = gpl_id.upper()
        stub = re.sub(r"\d{1,3}$", "nnn", gpl_id)
        url = f"https://ftp.ncbi.nlm.nih.gov/geo/platforms/{stub}/{gpl_id}/annot/{gpl_id}.annot.gz"

        dest = Path(destdir)
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / f"{gpl_id}.annot.gz"
        tmp = out.with_suffix(out.suffix + ".part")

        if out.exists() and not overwrite and out.stat().st_size > 0:
            return str(out)

        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        tmp.replace(out)
        return str(out)

    # ---------- helper: asegurar gpl.table (lazy) ----------
    def _ensure_gpl_with_table(gpl_id: str):
        gpl_id = gpl_id.upper()

        if gpl_id in _GPL_CACHE:
            return _GPL_CACHE[gpl_id]

        gpl0 = gse.gpls.get(gpl_id, None)
        if gpl0 is not None and hasattr(gpl0, "table") and gpl0.table is not None and not gpl0.table.empty:
            _GPL_CACHE[gpl_id] = gpl0
            return gpl0

        # HTTPS annot (si el fichero local está corrupto, reintenta overwrite=True)
        for ow in (False, True):
            try:
                annot_path = _download_gpl_annot_https(gpl_id, destdir=destdir, overwrite=ow)
                gpl1 = gp.get_GEO(filepath=annot_path, silent=True)
                if hasattr(gpl1, "table") and gpl1.table is not None and not gpl1.table.empty:
                    _GPL_CACHE[gpl_id] = gpl1
                    return gpl1
            except Exception:
                pass

        # fallback
        try:
            gpl2 = gp.get_GEO(geo=gpl_id, destdir=destdir, annotate_gpl=True, silent=True)
            if hasattr(gpl2, "table") and gpl2.table is not None and not gpl2.table.empty:
                _GPL_CACHE[gpl_id] = gpl2
                return gpl2
        except Exception:
            pass

        if gpl0 is None:
            raise RuntimeError(f"No se pudo cargar la GPL {gpl_id}.")
        _GPL_CACHE[gpl_id] = gpl0
        return gpl0

    # ---------- helper: adivinar columna de ID de sonda ----------
    def _guess_probe_id_col(gpl_table: pd.DataFrame, expr_index: pd.Index) -> str:
        expr_ids = pd.Index(expr_index.astype(str))

        candidates = [c for c in gpl_table.columns if str(c).strip().upper() in {"ID", "ID_REF"}]
        candidates += [c for c in gpl_table.columns if "probe" in str(c).lower()]
        if not candidates:
            candidates = list(gpl_table.columns[:3])

        best_col = str(gpl_table.columns[0])
        best_hits = -1
        for c in candidates:
            s = gpl_table[c].astype(str)
            hits = expr_ids.isin(pd.Index(s)).sum()
            if hits > best_hits:
                best_hits = hits
                best_col = c
        return best_col

    # ---------- helper: detectar ORF como Entrez (GPL8542) ----------
    def _find_orf_entrez_col(gpl_used) -> str | None:
        if not hasattr(gpl_used, "table") or gpl_used.table is None or gpl_used.table.empty:
            return None
        if "ORF" not in gpl_used.table.columns:
            return None
        s = pd.to_numeric(gpl_used.table["ORF"], errors="coerce")
        # si casi todo es numérico => muy probablemente Entrez ID
        return "ORF" if s.notna().mean() > 0.90 else None

    # ---------- helper: Entrez -> Symbol (MyGene.info) ----------
    def _entrez_to_symbol(entrez_ids, species="human", chunk_size=1000, retries=3, sleep_s=1.0):
        url = "https://mygene.info/v3/gene"
        out = {}

        ids = [str(x) for x in entrez_ids if pd.notna(x)]
        # usa cache
        ids_to_query = [i for i in ids if i not in _ENTREZ2SYM_CACHE]

        for i in range(0, len(ids_to_query), chunk_size):
            chunk = ids_to_query[i:i+chunk_size]

            last_err = None
            for att in range(retries):
                try:
                    r = requests.post(
                        url,
                        json={"ids": chunk, "fields": "symbol", "species": species},
                        timeout=60
                    )
                    r.raise_for_status()
                    data = r.json()
                    for item in data:
                        eid = str(item.get("_id") or item.get("query") or "")
                        sym = item.get("symbol", None)
                        if eid and sym:
                            _ENTREZ2SYM_CACHE[eid] = sym
                    last_err = None
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(sleep_s * (att + 1))

            if last_err is not None:
                # si falla la API, no abortamos: simplemente devolvemos lo que haya
                pass

        # construir mapping final con cache
        for eid in ids:
            sym = _ENTREZ2SYM_CACHE.get(str(eid))
            if sym:
                out[str(eid)] = sym
        return out

    # ---------- 2) Metadatos ----------
    gse_meta_df = get_gse_metadata(gse)
    gsm_meta_df = make_sample_metadata(gse)

    # ---------- 3) Expresión por sonda ----------
    first_gsm = next(iter(gse.gsms.values()))
    value_col = "VALUE" if "VALUE" in first_gsm.table.columns else \
        next(c for c in first_gsm.table.columns if pd.api.types.is_numeric_dtype(first_gsm.table[c]))

    expr_all = gse.pivot_samples(values=value_col)

    def gsm_ids_for_gpl(gpl_id: str):
        ids = []
        for gsm_id, gsm in gse.gsms.items():
            plat = _first_item(gsm.metadata, "platform_id", "")
            if plat == gpl_id:
                ids.append(gsm_id)
        return [x for x in ids if x in expr_all.columns]

    # ---------- 4) Por plataforma ----------
    by_gpl = {}
    gpl_ids = platforms if platforms else list(gse.gpls.keys())

    for gpl_id in gpl_ids:
        if gpl_id not in gse.gpls:
            continue

        gsm_ids = gsm_ids_for_gpl(gpl_id)
        expr_probes = expr_all[gsm_ids].copy() if gsm_ids else pd.DataFrame()

        did_log = False
        if not expr_probes.empty and _needs_log2(expr_probes):
            expr_probes = _apply_log2_like_R(expr_probes)
            did_log = True

        expr_genes = None
        gene_col = None
        probe_id_col = None
        n_mapped = 0
        n_symbol_mapped = 0
        gpl_used = gse.gpls[gpl_id]

        if annotate_to_gene and not expr_probes.empty:
            gpl_used = _ensure_gpl_with_table(gpl_id)

            # 1) intenta gene symbol “normal”
            gene_col = _find_gene_col(gpl_used)

            # 2) fallback: ORF (Entrez)
            if gene_col is None:
                gene_col = _find_orf_entrez_col(gpl_used)

            # 3) fallback: accession (GPL2700 -> GB_ACC)
            accession_col = None
            if gene_col is None:
                accession_col = _find_accession_col(gpl_used)

            # si no hay manera de anotar, saltamos
            if (gene_col is None) and (accession_col is None):
                pass
            else:
                gpl_tbl = gpl_used.table.copy()
                probe_id_col = _guess_probe_id_col(gpl_tbl, expr_probes.index)

                X = expr_probes.copy()
                X.index = X.index.astype(str)

                # =========================================
                # RAMA A) accession (GPL2700: GB_ACC)
                # =========================================
                if accession_col is not None:
                    map_df = gpl_tbl[[probe_id_col, accession_col]].dropna()
                    map_df[probe_id_col] = map_df[probe_id_col].astype(str)
                    map_df = map_df.drop_duplicates(subset=[probe_id_col], keep="first")

                    ann = X.join(map_df.set_index(probe_id_col), how="left")
                    n_mapped = int(ann[accession_col].notna().sum())

                    acc = ann[accession_col].map(_extract_first_accession)
                    acc_unique = pd.Index(acc.dropna().unique()).tolist()

                    m_acc2sym = _acc_to_symbol_mygene(acc_unique, species="human")
                    sym = acc.map(lambda a: m_acc2sym.get(a, np.nan))

                    n_symbol_mapped = int(sym.notna().sum())

                    key = sym if drop_unmapped_symbols else sym.fillna(acc)

                # =========================================
                # RAMA B) gene_col normal o ORF
                # =========================================
                else:
                    # map probe -> gene_col
                    map_df = gpl_tbl[[probe_id_col, gene_col]].dropna()
                    map_df[probe_id_col] = map_df[probe_id_col].astype(str)
                    map_df = map_df.drop_duplicates(subset=[probe_id_col], keep="first")

                    ann = X.join(map_df.set_index(probe_id_col), how="left")
                    n_mapped = int(ann[gene_col].notna().sum())

                    # ORF (Entrez) -> Symbol opcional
                    if gene_col == "ORF" and map_entrez_to_symbol:
                        entrez = pd.to_numeric(ann["ORF"], errors="coerce").astype("Int64")
                        entrez_str = entrez.astype(str)

                        m = _entrez_to_symbol(entrez_str.unique().tolist(), species="human")
                        sym = entrez_str.map(lambda eid: m.get(str(eid), np.nan))

                        n_symbol_mapped = int(sym.notna().sum())
                        key = sym if drop_unmapped_symbols else sym.fillna(entrez_str)
                    else:
                        key = ann[gene_col].map(_extract_first_symbol)
                        n_symbol_mapped = int(key.notna().sum())

                # =========================================
                # COLAPSO UNA SOLA VEZ (mean/median)
                # =========================================
                tmp = ann[gsm_ids].copy()
                tmp["__GENE__"] = key
                tmp = tmp.dropna(subset=["__GENE__"])

                if collapse == "mean":
                    expr_genes = tmp.groupby("__GENE__", dropna=True).mean(numeric_only=True).sort_index()
                elif collapse == "median":
                    expr_genes = tmp.groupby("__GENE__", dropna=True).median(numeric_only=True).sort_index()
                else:
                    raise ValueError("collapse debe ser 'mean' o 'median'")


        by_gpl[gpl_id] = {
            "gpl": gpl_used,
            "gsm_ids": gsm_ids,
            "expr_probes": expr_probes,
            "expr_genes": expr_genes,      # <- ahora ya sale en símbolos (si ORF+API ok)
            "log2_applied": did_log,
            "gene_col": gene_col,
            "probe_id_col": probe_id_col,
            "n_probes": int(expr_probes.shape[0]) if not expr_probes.empty else 0,
            "n_mapped": n_mapped,
            "n_symbol_mapped": n_symbol_mapped,
        }

    return gse, gse_meta_df, gsm_meta_df, by_gpl



def download_gse_family_soft_https(gse_id: str, destdir="geo", overwrite=False) -> str:
    gse_id = gse_id.upper()
    stub = re.sub(r"\d{1,3}$", "nnn", gse_id)  # GSE16011 -> GSE16nnn
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse_id}/soft/{gse_id}_family.soft.gz"

    dest = Path(destdir)
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / f"{gse_id}_family.soft.gz"
    tmp = out.with_suffix(out.suffix + ".part")

    if out.exists() and not overwrite and out.stat().st_size > 0:
            return str(out)

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    tmp.replace(out)
    return str(out)


def batch_load_with_metadata(
    jobs: Dict[str, List[str]],
    destdir: str = "geo",
    annotate_to_gene: bool = True,
    collapse: str = "mean",
):
    results: Dict[str, dict] = {}
    series_meta_rows = []
    sample_meta_rows = []

    for gse_id, gpls in jobs.items():
        try:
            gse, gse_meta, gsm_meta, by_gpl = load_everything_with_metadata(
                gse_id,
                destdir=destdir,
                platforms=gpls,
                annotate_to_gene=annotate_to_gene,
                collapse=collapse,
            )
            # guarda resultados por GSE
            results[gse_id] = {
                "gse": gse,
                "gse_meta": gse_meta,
                "gsm_meta": gsm_meta,
                "by_gpl": by_gpl,
            }
            # acumula metadatos “maestros”
            series_meta_rows.append(gse_meta)
            gm = gsm_meta.copy()
            gm.insert(0, "gse_accession", gse_id)
            sample_meta_rows.append(gm)

            print(f"[OK] 🎉 {gse_id}: {list(by_gpl.keys())}")

        except Exception as e:
            print(f"[ERROR] {gse_id}: {e}")

    series_meta = pd.concat(series_meta_rows, axis=0) if series_meta_rows else pd.DataFrame()
    sample_meta = pd.concat(sample_meta_rows, axis=0) if sample_meta_rows else pd.DataFrame()
    return {"results": results, "series_meta": series_meta, "sample_meta": sample_meta}




# Descargar GSE's Completos

In [ ]:
# Cohortes con una llamada molecular basada en expresión utilizable.
# GSE65194 se excluye deliberadamente: sus grupos son IHQ (incluido TNBC clínico).
jobs = {
    "GSE1456": ["GPL96"],
    "GSE162228": ["GPL570"],
    "GSE20711": ["GPL570"],
    "GSE21653": ["GPL570"],
    "GSE25066": ["GPL96"],
}

batch = batch_load_with_metadata(jobs)

for gse_id, info in batch["results"].items():
    for gpl_id, d in info["by_gpl"].items():
        globals()[f"expr_probes_{gse_id}"] = d["expr_probes"]
        globals()[f"expr_genes_{gse_id}"] = d["expr_genes"]
        print(f"creado expr_genes_{gse_id} y expr_probes_{gse_id}")

for gse_id, info in batch["results"].items():
    globals()[f"{gse_id}_meta"] = info["gsm_meta"]
    print(f"creado {gse_id}_meta")


# Seleccionar GSE's 

In [ ]:
pd.set_option("display.max_columns", None)

TARGET_CLASSES = ("LumA", "LumB", "HER2-enriched", "Basal-like")


def _find_metadata_column(meta, *candidates):
    # Encuentra una característica expandida o su columna GEO original.
    for candidate in candidates:
        if candidate in meta.columns:
            return candidate
    lower = {str(c).lower(): c for c in meta.columns}
    for candidate in candidates:
        cand = candidate.lower()
        for low, original in lower.items():
            if low.endswith("." + cand) or low.endswith("_" + cand):
                return original
    raise KeyError(f"No se encontró ninguna de estas columnas: {candidates}")


def _finish_intrinsic_cohort(
    meta,
    expr,
    *,
    cohort,
    raw_col,
    mapping,
    label_system,
    keep_mask=None,
    expected_counts=None,
):
    # Filtra, armoniza, documenta y audita una cohorte molecular.
    out = meta.copy()
    if keep_mask is not None:
        keep_mask = pd.Series(keep_mask, index=out.index).fillna(False).astype(bool)
        out = out.loc[keep_mask].copy()

    out["label_raw"] = out[raw_col].astype("string").str.strip()
    out["label"] = out["label_raw"].map(mapping)
    out = out[out["label"].isin(TARGET_CLASSES)].copy()
    out["batch"] = cohort
    out["sample"] = out["geo_accession"].astype(str)
    out["label_source"] = raw_col
    out["label_system"] = label_system
    out["sample_role"] = "primary_breast_tumor"

    if out.index.duplicated().any():
        raise RuntimeError(f"{cohort}: GSM duplicados tras el filtrado")
    if out["sample"].duplicated().any():
        raise RuntimeError(f"{cohort}: identificadores de muestra duplicados")

    expr_out = expr.loc[:, out.index].copy()
    if list(expr_out.columns.astype(str)) != list(out.index.astype(str)):
        raise RuntimeError(f"{cohort}: expresión y metadata no quedaron alineadas")

    counts = out["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0).astype(int)
    if expected_counts is not None and counts.to_dict() != expected_counts:
        raise RuntimeError(
            f"{cohort}: conteos distintos de los auditados. "
            f"Observado={counts.to_dict()}, esperado={expected_counts}"
        )
    print(f"\n{cohort}: {len(out)} tumores")
    display(counts.rename("n"))
    return out, expr_out


## GSE1456 🎉

In [ ]:
raw_col = _find_metadata_column(
    GSE1456_meta,
    "subtype",
    "characteristics_ch1.6.SUBTYPE",
)
platform_mask = GSE1456_meta["platform_id"].astype(str).eq("GPL96")

GSE1456_meta, expr_genes_GSE1456 = _finish_intrinsic_cohort(
    GSE1456_meta,
    expr_genes_GSE1456,
    cohort="GSE1456",
    raw_col=raw_col,
    mapping={
        "Luminal A": "LumA",
        "Luminal B": "LumB",
        "ERBB2": "HER2-enriched",
        "Basal": "Basal-like",
    },
    label_system="published_intrinsic_SSP_expression_call",
    keep_mask=platform_mask,
    expected_counts={
        "LumA": 39,
        "LumB": 23,
        "HER2-enriched": 15,
        "Basal-like": 25,
    },
)


In [ ]:
display(GSE1456_meta[["sample", "label_raw", "label", "label_system"]].head())


In [ ]:
print(expr_genes_GSE1456.shape[1], GSE1456_meta.shape[0])

## GSE162228 🎉

In [ ]:
raw_col = _find_metadata_column(
    GSE162228_meta,
    "array_ssp2",
    "characteristics_ch1.10.array_ssp2",
)

# GSE162228 contiene 109 tumores y 24 tejidos adyacentes normales. SSP2 fuerza
# una de cuatro etiquetas incluso para esos 24 normales, por lo que hay que
# retirarlos por identidad tisular antes de mapear la clase.
title = GSE162228_meta["title"].astype(str)
is_adjacent_normal = (
    title.str.contains("normal", case=False, regex=False, na=False)
    | title.str.contains(r"_\s*\d+N(?:\b|$)", case=False, regex=True, na=False)
)
if int(is_adjacent_normal.sum()) != 24:
    raise RuntimeError(
        "GSE162228: se esperaban 24 tejidos normales adyacentes; "
        f"se detectaron {int(is_adjacent_normal.sum())}."
    )

GSE162228_meta, expr_genes_GSE162228 = _finish_intrinsic_cohort(
    GSE162228_meta,
    expr_genes_GSE162228,
    cohort="GSE162228",
    raw_col=raw_col,
    mapping={
        "Lum-A": "LumA",
        "Lum-B": "LumB",
        "HER2": "HER2-enriched",
        "Basal": "Basal-like",
    },
    label_system="array_SSP2_expression_call",
    keep_mask=~is_adjacent_normal,
    expected_counts={
        "LumA": 33,
        "LumB": 39,
        "HER2-enriched": 21,
        "Basal-like": 16,
    },
)


In [ ]:
display(GSE162228_meta[["sample", "label_raw", "label", "label_system"]].head())


In [ ]:
print(expr_genes_GSE162228.shape[1], GSE162228_meta.shape[0])

##  GSE20711 🎉

In [ ]:
# Se usa la llamada basada en expresión (subtypege), no el sustituto IHQ.
raw_col = _find_metadata_column(
    GSE20711_meta,
    "subtypege",
    "characteristics_ch1.9.subtypege",
)

GSE20711_meta, expr_genes_GSE20711 = _finish_intrinsic_cohort(
    GSE20711_meta,
    expr_genes_GSE20711,
    cohort="GSE20711",
    raw_col=raw_col,
    mapping={
        "LumA": "LumA",
        "LumB": "LumB",
        "HER2+": "HER2-enriched",
        "Basal": "Basal-like",
    },
    label_system="published_gene_expression_subtype_call",
    expected_counts={
        "LumA": 23,
        "LumB": 22,
        "HER2-enriched": 21,
        "Basal-like": 22,
    },
)


In [ ]:
display(GSE20711_meta[["sample", "label_raw", "label", "label_system"]].head())


In [ ]:
print(expr_genes_GSE20711.shape[1], GSE20711_meta.shape[0])

##  GSE21653 🎉

In [ ]:
# GEO distribuye "molecular subtype" en varias columnas de characteristics.
# Se coalescen todas; elegir una sola pierde 17 tumores válidos de GSE21653.
mol_cols = [
    col for col in GSE21653_meta.columns
    if "molecular subtype" in str(col).lower().replace("_", " ")
]
if not mol_cols:
    raise KeyError("GSE21653: no se encontró ninguna columna molecular subtype")

mol = GSE21653_meta[mol_cols].copy()
mol = mol.replace(r"^\s*$", pd.NA, regex=True)
mol = mol.apply(lambda col: col.astype("string").str.strip())
conflict = mol.apply(
    lambda row: len(set(row.dropna().astype(str))) > 1,
    axis=1,
)
if conflict.any():
    examples = mol.loc[conflict].head().to_dict(orient="index")
    raise RuntimeError(
        f"GSE21653: columnas de subtipo en conflicto para {int(conflict.sum())} "
        f"muestras; ejemplos={examples}"
    )
GSE21653_meta["molecular_subtype_coalesced"] = mol.bfill(axis=1).iloc[:, 0]

GSE21653_meta, expr_genes_GSE21653 = _finish_intrinsic_cohort(
    GSE21653_meta,
    expr_genes_GSE21653,
    cohort="GSE21653",
    raw_col="molecular_subtype_coalesced",
    mapping={
        "LuminalA": "LumA",
        "LuminalB": "LumB",
        "ERBB2": "HER2-enriched",
        "Basal": "Basal-like",
    },
    label_system="published_molecular_expression_subtype_call",
    expected_counts={
        "LumA": 89,
        "LumB": 49,
        "HER2-enriched": 24,
        "Basal-like": 75,
    },
)


In [ ]:
display(GSE21653_meta[["sample", "label_raw", "label", "label_system"]].head())


In [ ]:
print(expr_genes_GSE21653.shape[1], GSE21653_meta.shape[0])

##  GSE25066 🎉

In [ ]:
raw_col = _find_metadata_column(
    GSE25066_meta,
    "pam50_class",
    "characteristics_ch1.20.pam50_class",
)

GSE25066_meta, expr_genes_GSE25066 = _finish_intrinsic_cohort(
    GSE25066_meta,
    expr_genes_GSE25066,
    cohort="GSE25066",
    raw_col=raw_col,
    mapping={
        "LumA": "LumA",
        "LumB": "LumB",
        "Her2": "HER2-enriched",
        "Basal": "Basal-like",
    },
    label_system="PAM50_expression_call",
    expected_counts={
        "LumA": 160,
        "LumB": 78,
        "HER2-enriched": 37,
        "Basal-like": 189,
    },
)


In [ ]:
display(GSE25066_meta[["sample", "label_raw", "label", "label_system"]].head())


In [ ]:
print(expr_genes_GSE25066.shape[1], GSE25066_meta.shape[0])

## GSE65194 · excluida del endpoint intrínseco principal

GSE65194 contiene 130 tumores únicos clasificados mediante grupos **IHQ**
(Luminal A, Luminal B, HER2 y TNBC), además de 23 réplicas técnicas, tejidos
sanos y líneas celulares. Es útil para un análisis clínico separado, pero sus
etiquetas no son intercambiables con LumA, LumB, HER2-enriched y Basal-like
basados en expresión. Por eso no se descarga ni se mezcla aquí.


# Unión de las cinco cohortes de microarrays moleculares


In [ ]:
import pandas as pd

INCLUDED_MICROARRAY_COHORTS = (
    "GSE1456",
    "GSE162228",
    "GSE20711",
    "GSE21653",
    "GSE25066",
)
PREFER_INTERSECTION = True
MIN_COMMON_GENES_WARN = 1000


def build_intrinsic_microarray_tables(globs, study_ids):
    expr_list = []
    meta_list = []

    for study_id in study_ids:
        expr_name = f"expr_genes_{study_id}"
        meta_name = f"{study_id}_meta"
        if expr_name not in globs or meta_name not in globs:
            raise RuntimeError(f"Falta {expr_name} o {meta_name}")

        X = globs[expr_name].copy()
        M = globs[meta_name].copy()
        X.columns = X.columns.astype(str)
        M["sample"] = M["sample"].astype(str)

        if X.columns.duplicated().any() or M["sample"].duplicated().any():
            raise RuntimeError(f"{study_id}: muestras duplicadas")
        if set(X.columns) != set(M["sample"]):
            only_x = sorted(set(X.columns) - set(M["sample"]))[:5]
            only_m = sorted(set(M["sample"]) - set(X.columns))[:5]
            raise RuntimeError(
                f"{study_id}: expresión/metadata no coinciden; "
                f"solo expresión={only_x}, solo metadata={only_m}"
            )

        order = M["sample"].tolist()
        X = X.loc[:, order]
        prefixed = [f"{study_id}:{sample}" for sample in order]
        X.columns = prefixed
        M["sample"] = prefixed
        M["batch"] = study_id

        expr_list.append(X)
        meta_list.append(M)

    gene_sets = [set(x.index.astype(str)) for x in expr_list]
    if PREFER_INTERSECTION:
        genes = sorted(set.intersection(*gene_sets))
        if len(genes) < MIN_COMMON_GENES_WARN:
            print(f"AVISO: solo hay {len(genes)} genes comunes")
        aligned = []
        for X in expr_list:
            X = X.copy()
            X.index = X.index.astype(str)
            aligned.append(X.loc[genes])
    else:
        genes = sorted(set.union(*gene_sets))
        aligned = []
        for X in expr_list:
            X = X.copy()
            X.index = X.index.astype(str)
            aligned.append(X.reindex(genes))

    expr_mega = pd.concat(aligned, axis=1)
    meta_mega = pd.concat(meta_list, axis=0, ignore_index=True)
    if expr_mega.columns.duplicated().any() or meta_mega["sample"].duplicated().any():
        raise RuntimeError("Hay muestras duplicadas tras la unión")
    if list(expr_mega.columns) != meta_mega["sample"].tolist():
        raise RuntimeError("La unión final no conserva el mismo orden en expresión y metadata")

    print(f"Genes comunes: {expr_mega.shape[0]}")
    print(f"Tumores: {expr_mega.shape[1]}")
    print(f"Cohortes: {meta_mega['batch'].nunique()}")
    return expr_mega, meta_mega


expr_genes_microarrays, microarrays_meta = build_intrinsic_microarray_tables(
    globals(), INCLUDED_MICROARRAY_COHORTS
)


In [ ]:
provenance_cols = [
    "sample",
    "batch",
    "label",
    "label_raw",
    "label_source",
    "label_system",
    "sample_role",
]
microarrays_meta = microarrays_meta[provenance_cols].copy()
display(microarrays_meta["batch"].value_counts().sort_index())


In [ ]:
# TOTAL de NaN en todo el DataFrame
total_nans = expr_genes_microarrays.isna().sum().sum()

# NaN POR COLUMNA (ordenado de mayor a menor)
nans_por_col = expr_genes_microarrays.isna().sum().sort_values(ascending=False)

# Porcentaje de NaN POR COLUMNA
pct_nans_por_col = (expr_genes_microarrays.isna().mean() * 100).sort_values(ascending=False)

# Número de columnas que tienen al menos un NaN
num_cols_con_nan = expr_genes_microarrays.isna().any().sum()

# Columnas que tienen NaN
cols_con_nan = expr_genes_microarrays.columns[expr_genes_microarrays.isna().any()].tolist()

# NaN POR FILA
nans_por_fila = expr_genes_microarrays.isna().sum(axis=1)

print("Total NaN:", total_nans)
print("Columnas con NaN:", num_cols_con_nan)
print("Top columnas con NaN:\n", nans_por_col.head())

In [ ]:
n_before = len(expr_genes_microarrays)
expr_genes_microarrays = expr_genes_microarrays.dropna(axis=0, how="any")
print("Filas eliminadas:", n_before - len(expr_genes_microarrays))

In [ ]:
expr_genes_microarrays.to_csv("./Data/expr_genes_microarrays.csv")
microarrays_meta.to_csv("./Data/microarrays_meta.csv", index=False)
print("Guardados Data/expr_genes_microarrays.csv y Data/microarrays_meta.csv")


In [ ]:
EXPECTED_COHORTS = set(INCLUDED_MICROARRAY_COHORTS)
observed_cohorts = set(microarrays_meta["batch"].astype(str))
assert observed_cohorts == EXPECTED_COHORTS, (observed_cohorts, EXPECTED_COHORTS)
assert set(microarrays_meta["label"].dropna()) == set(TARGET_CLASSES)
assert microarrays_meta["sample"].is_unique
assert expr_genes_microarrays.columns.is_unique
assert list(expr_genes_microarrays.columns) == microarrays_meta["sample"].tolist()

n_labels = microarrays_meta.groupby("batch")["label"].nunique()
assert (n_labels == 4).all(), n_labels.to_dict()
display(n_labels.sort_index().rename("n_labels"))
print("Microarrays moleculares guardados:", expr_genes_microarrays.shape)


In [ ]:
class_by_cohort = pd.crosstab(
    microarrays_meta["batch"],
    microarrays_meta["label"],
).reindex(columns=TARGET_CLASSES, fill_value=0)
display(class_by_cohort)


## Auditoría final esperada

La descarga vigente de GEO debe producir 1.000 tumores de microarray:
344 LumA, 211 LumB, 118 HER2-enriched y 327 Basal-like. Si GEO cambia sus
metadatos, el notebook se detendrá en vez de aceptar silenciosamente otro
conjunto de muestras.


In [ ]:
EXPECTED_TOTAL_COUNTS = {
    "LumA": 344,
    "LumB": 211,
    "HER2-enriched": 118,
    "Basal-like": 327,
}
observed_total_counts = (
    microarrays_meta["label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .astype(int)
    .to_dict()
)
assert observed_total_counts == EXPECTED_TOTAL_COUNTS, (
    observed_total_counts,
    EXPECTED_TOTAL_COUNTS,
)
assert len(microarrays_meta) == 1000
print("Auditoría microarrays correcta:", observed_total_counts)
